# KTO training on ChartQA (method 4/7)

Same trainer, hyperparameters (v14 config: lr=2e-6, beta=0.1, 2 epochs,
warn-only collapse guard, balanced 1:3 desirable:undesirable) as CharXiv's KTO
run -- only the dataset (`experiments/020_chartqa_transfer/data/kto_samples.jsonl`,
147 pos / 465 neg before balancing) and image directory change.


In [ ]:
import subprocess, sys

gpu_names = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip().splitlines()
print('Detected GPUs (nvidia-smi):', gpu_names)
is_p100 = any('P100' in n for n in gpu_names)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

if is_p100:
    print('*** Tesla P100 detected. Installing the validated older stack...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow'], check=True)
else:
    print('Non-P100 GPU: upgrading transformers to a current release, leaving torch/peft/accelerate at image defaults.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.49.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft==0.14.0', 'qwen-vl-utils==0.0.14'], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, transformers: {transformers.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')


In [ ]:
import os
from pathlib import Path
import subprocess

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')


In [ ]:
import sys

env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/train_kto.py',
    '--dataset-path', 'experiments/020_chartqa_transfer/data/kto_samples.jsonl',
    '--images-dir', 'data/ChartQA/images',
    '--output-dir', '/kaggle/working/qwen_vl_kto_chartqa_adapter',
    '--epochs', '2',
    '--batch-size', '1',
    '--lr', '2e-6',
    '--beta', '0.1',
    '--max-logp-drop', '55',
    '--collapse-guard-warn-only',
    '--balance-kto',
    '--auto-desirable-weight',
]
subprocess.run(cmd, env=env, check=True)


In [ ]:
out_dir = Path('/kaggle/working/qwen_vl_kto_chartqa_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'KTO ChartQA adapter directory {out_dir} contents: {files}')
